# LMP-SPARK
**Author:** Ryan J. McLaughlin  
**Date:** 2025-05-06

This is meant to be a complete end-to-end document for running:

1. Amplicon Sequence Variant (ASV) pipeline
2. General statistics
3. Downstream analytics
4. Figure/Table creation

## 1. Amplicon Sequence Variant (ASV) pipeline
This section reviews the steps involved in creating ASVs from raw FASTQ data.

### Setup Environments for running the pipeline

In [ ]:
%%bash
# Define the environment name
ENV_NAME="spark_env"
ENV_YAML="$PWD/spark_env.yaml"

QI_NAME="qiime2-amplicon-2024.10"

# Check if the environment exists
if mamba env list | grep -q "^${ENV_NAME} "; then
    echo "Environment ${ENV_NAME} already exists."
else
    echo "Environment ${ENV_NAME} does not exist. Creating it..."
    mamba env create -y -n ${ENV_NAME} -f ${ENV_YAML}
fi

# Check if the QIIME2 environment exists
if mamba env list | grep -q "^${QI_NAME} "; then
    echo "Environment ${QI_NAME} already exists."
else
    echo "Environment ${QI_NAME} does not exist. Creating it..."
    mamba env create -y -n ${QI_NAME} -c bioconda qiime2-amplicon-2024.10
fi

### Run the ASV pipeline

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

./run_vsearch.sh

### Run General Statistics

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
THREADS="$(nproc)"

seqkit stat -a -T -o ../final_output/stats/fastq_stats.tsv -j ${THREADS} ../fastq_combined/*.fastq.gz
seqkit stat -a -T -o ../final_output/stats/fastp_fastqs.tsv -j ${THREADS} ../final_output/fastp/*.fastq.gz
seqkit stat -a -T -o ../final_output/stats/filtered_fastqs.tsv -j ${THREADS} ../final_output/filtered/*.fasta
seqkit stat -a -T -o ../final_output/stats/concat_fastas.tsv -j ${THREADS} ../final_output/concat/concat.fasta

### Run QIIME2 Taxonomic Classifier

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate qiime2-amplicon-2024.10
mkdir -p ../final_output/taxonomy
awk '/^>/ {print; next} {print toupper($0)}' ../final_output/ASVs/ASVs.fasta > ../final_output/ASVs/ASVs.upper.fasta
python qiime_vs_classifier.py \
  --input-fasta ../final_output/ASVs/ASVs.upper.fasta \
  --ref-taxonomy ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-tax.qza \
  --ref-seqs ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-seqs-DNA.qza \
  --output-tsv ../final_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
  --stats-output ../final_output/taxonomy/ASV_SILVA_stats.full-length.vsearch.tsv

### Mitomaster, decontamination, mitoDB

In [ ]:
mkdir -p ../final_output/mitomap
seqkit split -s 100 -O ../final_output/ASVs/chunks ../final_output/ASVs/ASVs.fasta
python ./mitomaster.py
blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/final_output/ASVs/ASVs.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/mito_ncbi \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/final_output/mitomap/mito_ncbi.blast6.tsv

blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/final_output/ASVs/ASVs.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/ssu_pipeline_contaminants \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/final_output/mitomap/ssu_pipeline_contaminants.blast6.tsv
python ./mito_checker.py

### Filter ASV count tables

In [ ]:
python filter_ASV_table.py \
    ~/SeqData/SeqData/UBC/LMP_priority1/final_output/ASVs/ASV_counts.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/final_output/mitomap/nontarget.master.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/final_output/ASVs/ASV_filtered.tsv \
    1000 \
    0.005

### Build Sankey Diagram

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

mkdir -p ../final_output/metadata
python sankey_builder.py

### Plot Metadata

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_metadata.py

### Plot Upset

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_upset.py
python venn_bubbles.py

### Run Alpha and Beta Diversity

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python calc_div.py
python plot_diversity.py

### Run indicspecies (R)

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
Rscript run_indicspecies.R

### Plot indicspecies Results

In [19]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_indicspecies.py

/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['log_p'] = -np.log10(df['p.value'])
/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['significance'] = False  # Default color for non-significant
/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

(36, 18)
(11, 18)


/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['type_color'] = [x if y == True else 'lightgrey' for x,y in zip(df['type_color'], df['status_significance'])]
/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:175: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['type_color'] = ['lightgrey' if ((y == True) & (x == 'lightgrey')) else x


(21, 18)
(9, 18)


### Plot Clustermaps

In [20]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_clustermaps.py

### Run SPIEC-EASI (R)

In [12]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

Rscript run_spieceasi.R


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Attaching package: ‘igraph’

The following object is masked from ‘package:SpiecEasi’:

    make_graph

The following object is masked from ‘package:tidyr’:

    crossing

The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union

The following object is masked from ‘package:tibble’:

    as_data_frame

The following object is masked from ‘package:rlang’:

    is_named

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union

── Attaching core tidyverse packages ───────────────────��──── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ purrr     1.0.4
✔ lubridate 1.9.4     ✔ stringr   1.5.1
── Conflicts ───────────────────────────��────────────── tidyverse_conflicts() ──
✖ lubrida

[1] "Loaded filtered count data from /home/ryan/Projects/UBC/LMP/SPARK_data/vsearch_output/spieceasi/count_data_filtered.RDS"
     ASV558 ASV46 ASV126 ASV187 ASV522 ASV374 ASV208 ASV358 ASV537 ASV372 ASV40
[1,]      0    31      0      0      0      0      0      0      0      0     2
[2,]      0     0      0      0      0      0      0      0      0      0     0
[3,]      0     3    251      0      0      0      0      0      0      0     0
[4,]      0     7    158      0      0      0      0      0      0      0     0
[5,]     47     8      1      0      0      0      0      0      3      0     9
[6,]      0     0      0      0      0      0      0      0      0      0     8
     ASV307 ASV84 ASV73 ASV232 ASV390 ASV370 ASV100 ASV473 ASV351 ASV87 ASV59
[1,]      0     0    11      6      0      0      0      0      0     0     0
[2,]      0     0     0      0      0      0      0      0      0     0     0
[3,]      0     0     0      0      0      0      0      0      0     2     0
[4

Error: object 'flurp' not found
Execution halted


CalledProcessError: Command 'b'MDIR=$(dirname $(which mamba))\nsource ${MDIR}/../etc/profile.d/conda.sh\nconda activate spark_env\n\nRscript run_spieceasi.R\n'' returned non-zero exit status 1.

### Graph Network

In [29]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python graph_network.py

Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
/home/ryan/Projects/UBC/LMP/SPARK/graph_network.py:533: UserWarning: 

The connectionstyle keyword argument is not applicable when drawing edges
with LineCollection.

To make this warning go away, either specify `arrows=True` to
force FancyArrowPatches or use the default values.
Note that using FancyArrowPatches may be slow for large graphs.

  nx.draw_networkx_edges(G, pos,
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
/home/ryan/Projects/UBC/LMP/SPARK/graph_network.py:596: UserWarning: 

The connectionstyle keyword argument is not applicable when

CalledProcessError: Command 'b'MDIR=$(dirname $(which mamba))\nsource ${MDIR}/../etc/profile.d/conda.sh\nconda activate spark_env\n\npython graph_network.py\n'' returned non-zero exit status 1.

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

Rscript run_spieceasi_multi.R


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Attaching package: ‘igraph’

The following object is masked from ‘package:SpiecEasi’:

    make_graph

The following object is masked from ‘package:tidyr’:

    crossing

The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union

The following object is masked from ‘package:tibble’:

    as_data_frame

The following object is masked from ‘package:rlang’:

    is_named

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union

── Attaching core tidyverse packages ───────────────────��──── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ purrr     1.0.4
✔ lubridate 1.9.4     ✔ stringr   1.5.1
── Conflicts ───────────────────────────��────────────── tidyverse_conflicts() ──
✖ lubrida

[1] "Count Data:"
[1] 465 129
[1] "Count Data:"
[1] 465 129


Processing patient: P17


[1] "Sub Count Data:"
[1] 465   5


Applying data transformations...
Selecting model with pulsar using stars...
